# 05 - Sensitivity analysis: which parameters matter most?

A tornado-diagram style sensitivity analysis. For each key parameter, we vary it across a credible range while holding everything else fixed at the post_genai_2026 baseline, and observe the effect on the inversion premium.

This isolates the parameters whose calibration most strongly affects the paper's central conclusion.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.simulation import load_scenario, run_single_simulation
from copy import deepcopy
import numpy as np
import matplotlib.pyplot as plt

base_config = load_scenario('../config/scenarios/post_genai_2026.yaml',
                            '../config/parameters.yaml')

def run_with_param(param_path, value):
    cfg = deepcopy(base_config)
    keys = param_path.split('.')
    target = cfg
    for k in keys[:-1]:
        target = target[k]
    target[keys[-1]] = value
    result = run_single_simulation(cfg)
    classical = result.valuations_at_exit['damodaran_classical']
    inverted = result.valuations_at_exit['damodaran_inverted']
    return 100 * (inverted - classical) / max(classical, 1.0)

# Sweep ai_substitution_potential_layer3
ai_values = np.linspace(0.10, 0.95, 18)
ai_results = [run_with_param('startup.ai_substitution_potential_layer3', v) for v in ai_values]

# Sweep team_layer3_share by editing team_layer_distribution
shares = np.linspace(0.30, 0.85, 12)
share_results = []
for s in shares:
    cfg = deepcopy(base_config)
    cfg['startup']['team_layer_distribution'] = {'layer_3': s, 'layer_4': (1-s)*0.7, 'layer_5': (1-s)*0.3}
    res = run_single_simulation(cfg)
    classical = res.valuations_at_exit['damodaran_classical']
    inverted = res.valuations_at_exit['damodaran_inverted']
    share_results.append(100 * (inverted - classical) / max(classical, 1.0))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(ai_values, ai_results, 'o-', color='#e74c3c')
axes[0].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[0].set_xlabel('ai_substitution_potential_layer3')
axes[0].set_ylabel('Inversion premium (%)')
axes[0].set_title('Sensitivity to AI substitutability')
axes[1].plot(shares, share_results, 'o-', color='#3498db')
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1].axvline(base_config['valuation']['damodaran_inverted_threshold_layer3_share'],
                color='gray', linewidth=0.6, linestyle=':')
axes[1].set_xlabel('team_layer3_share')
axes[1].set_ylabel('Inversion premium (%)')
axes[1].set_title('Sensitivity to team composition')
plt.tight_layout(); plt.show()